# Ekstraksi 68 Fitur TSFEL pada Data CO Lenteng

## 1. Instalasi dan Persiapan Library openEO

Program diawali dengan melakukan instalasi library openeo.

Library ini digunakan untuk mengakses dan mengambil data pengamatan bumi dari layanan openEO.

Setelah library terpasang, program melakukan import dan membuat koneksi ke Copernicus Data Space Ecosystem.

Mengimpor library openeo.
Membuat koneksi ke server openEO menggunakan autentikasi OIDC.

Autentikasi diperlukan agar program dapat mengakses koleksi data Sentinel-5P yang digunakan dalam penelitian. Pada notebook, proses autentikasi berhasil dilakukan menggunakan device code flow.

In [1]:
!pip install openeo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


In [2]:
import openeo
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=MOHH-DVCU 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


## 2. Menentukan Area Observasi

Program kemudian menentukan area yang akan diamati menggunakan koordinat geografis.

Variabel aoi merupakan singkatan dari Area of Interest, yaitu wilayah yang menjadi fokus penelitian.

Koordinat tersebut membentuk sebuah polygon yang digunakan untuk menentukan wilayah pengamatan. Dengan demikian, data yang diperoleh nantinya dapat difokuskan pada wilayah yang telah ditentukan.


Setelah area pengamatan ditentukan, program mengambil data dari koleksi.
Data yang digunakan merupakan data Carbon Monoxide (CO) dengan rentang waktu sekitar satu tahun.

Dengan menentukan bands=["CO"], program hanya mengambil variabel yang berkaitan dengan konsentrasi karbon monoksida sehingga data yang diproses sesuai dengan tujuan penelitian.

Data CO kemudian diagregasi berdasarkan waktu menggunakan parameter:

- reducer="mean" digunakan untuk menghitung nilai rata-rata.
- period="day" digunakan agar data dikelompokkan berdasarkan hari.

Dengan proses ini, data yang sebelumnya dapat memiliki banyak nilai dalam suatu periode kemudian diringkas menjadi representasi harian.

Selanjutnya program melakukan agregasi spasial untuk menghitung nilai rata-rata CO pada area yang telah ditentukan oleh aoi.

Jadi, proses pengolahan data pada tahap ini dapat diringkas

In [3]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [113.7130839,-6.9970047],
            [113.7109033,-7.0548198],
            [113.8348847,-7.0560564],
            [113.8348847,-6.9994782],
            [113.7130839,-6.9970047]
        ]
    ]
}

s5post = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent={
        "west": 112.68,
        "south": -7.20,
        "east": 113.09,
        "north": -6.89
    },
    bands=["CO"],
)

s5p_no2_daily = s5post.aggregate_temporal_period(reducer="mean", period="day")

s5p_no2_aoi = s5p_no2_daily.aggregate_spatial(reducer="mean", geometries=aoi)

## 3 Membuat Visualisasi Area Observasi

Program menggunakan library folium untuk menampilkan wilayah penelitian pada peta.

In [24]:
import folium

lats = [coord[1] for coord in aoi["coordinates"][0]]
lons = [coord[0] for coord in aoi["coordinates"][0]]
center_lat = sum(lats) / len(lats)
center_lon = sum(lons) / len(lons)

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')

folium_coords = [[coord[1], coord[0]] for coord in aoi["coordinates"][0]]

folium.Polygon(
    locations=folium_coords,
    color='red',
    weight=3,
    fill=True,
    fill_color='blue',
    fill_opacity=0.2,
    tooltip='Area Observasi (Sumenep)'
).add_to(m)

m

## 4 Menjalankan Proses Pengambilan Data

Data kemudian diproses menggunakan execute_batch():

Perintah tersebut menjalankan proses pengambilan data secara batch dan menyimpannya ke dalam file:

COinLenteng.nc

Format .nc merupakan format NetCDF yang umum digunakan untuk menyimpan data ilmiah dan data lingkungan.

In [5]:
job = s5post.execute_batch(title="CO in Sumenep", outputfile="COinLenteng.nc")

0:00:00 Job 'j-2609132325294bb8b9d2493f37cdff4e': send 'start'
0:00:03 Job 'j-2609132325294bb8b9d2493f37cdff4e': queued (progress 0%)
0:00:08 Job 'j-2609132325294bb8b9d2493f37cdff4e': queued (progress 0%)
0:00:15 Job 'j-2609132325294bb8b9d2493f37cdff4e': queued (progress 0%)
0:00:23 Job 'j-2609132325294bb8b9d2493f37cdff4e': queued (progress 0%)
0:00:33 Job 'j-2609132325294bb8b9d2493f37cdff4e': queued (progress 0%)
0:00:45 Job 'j-2609132325294bb8b9d2493f37cdff4e': running (progress N/A)
0:01:01 Job 'j-2609132325294bb8b9d2493f37cdff4e': running (progress N/A)
0:01:20 Job 'j-2609132325294bb8b9d2493f37cdff4e': running (progress N/A)
0:01:44 Job 'j-2609132325294bb8b9d2493f37cdff4e': running (progress N/A)
0:02:15 Job 'j-2609132325294bb8b9d2493f37cdff4e': running (progress N/A)
0:02:52 Job 'j-2609132325294bb8b9d2493f37cdff4e': running (progress N/A)
0:03:39 Job 'j-2609132325294bb8b9d2493f37cdff4e': running (progress N/A)
0:04:38 Job 'j-2609132325294bb8b9d2493f37cdff4e': finished (progress 10

## 5 Membaca File NetCDF

Untuk membaca file NetCDF, perlu untuk memasang library netCDF4.
setelah NetCDF Kemudian file dibuka:

Program mengambil variabel CO dan waktu, Selanjutnya jumlah data dihitung untuk menunjukkan hasil yang di peroleh.

In [6]:
!pip install netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.3 MB/s eta 0:00:00


In [7]:
import netCDF4
import os

file_path = "/content/COinLenteng.nc"
ds = netCDF4.Dataset(file_path)

co = ds.variables["CO"][:]
time = ds.variables["t"][:]
print("Total Data = ", len(co), "hari")

Total Data =  365 hari


## 6 Membentuk DataFrame

Data CO kemudian diproses menggunakan pandas dan numpy.

Data CO dikonversi menjadi array. Apabila data memiliki lebih dari satu dimensi, program menghitung rata-rata nilai pada setiap hari,
Tujuan bagian ini adalah memastikan data yang digunakan menjadi satu nilai CO untuk setiap hari.

Kemudian dibuat rentang tanggal,
Setelah itu tanggal dan nilai CO digabungkan menjadi DataFrame. Hasilnya berupa tabel dengan dua kolom utama yaitu:

- Date → tanggal pengamatan.
- CO_Concentration → konsentrasi CO.

In [8]:
import pandas as pd
import numpy as np

co_array = np.array(co)
num_days = co_array.shape[0]

if co_array.ndim > 1:
    co_data_daily_mean = np.mean(co_array.reshape(num_days, -1), axis=1)
else:
    co_data_daily_mean = co_array.flatten()

start_date = pd.Timestamp('2025-08-24') # From temporal_extent=['2025-08-24', '2026-08-24']
date_range = pd.date_range(start=start_date, periods=num_days, freq='D')

df = pd.DataFrame({
    'Date': date_range,
    'CO_Concentration': co_data_daily_mean
})

display(df)

,Date,CO_Concentration
0,2025-08-24,0.018319
1,2025-08-25,0.020895
2,2025-08-26,0.000000
3,2025-08-27,0.026487
4,2025-08-28,0.010172
...,...,...
360,2026-08-19,0.018558
361,2026-08-20,0.022621
362,2026-08-21,0.021804
363,2026-08-22,0.022201


## 7. Menangani Missing Value

Tahap berikutnya adalah pemeriksaan Missing Value.

Setelah jumlah Missing Value diperiksa, program melakukan interpolasi linear:

Interpolasi linear digunakan untuk memperkirakan nilai yang hilang berdasarkan nilai sebelum dan sesudahnya.

Kemudian program kembali menghitung jumlah missing value untuk memastikan hasilnya.

In [9]:
# 1. Missing Value
print(f"Missing values before imputation: {df['CO_Concentration'].isnull().sum()}")
df['CO_Concentration'] = df['CO_Concentration'].interpolate(method='linear')
print(f"Missing values after imputation: {df['CO_Concentration'].isnull().sum()}")

Missing values before imputation: 0
Missing values after imputation: 0


## 8. Mendeteksi dan Menangani Outlier

Program berikutnya melakukan pemeriksaan terhadap data pencilan atau outlier.

Metode yang digunakan adalah Interquartile Range (IQR).

Pertama, program menghitung kuartil pertama dan ketiga:

Kemudian menghitung IQR:

IQR = Q3 - Q1

Batas bawah dan batas atas ditentukan menggunakan:

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

Nilai yang berada di luar batas tersebut dianggap sebagai outlier.

Apabila terdapat outlier, program menggantinya dengan nilai median dari data yang tidak termasuk outlier, Kemudian nilai outlier diganti.

In [10]:
# 2. Outlier

Q1 = df['CO_Concentration'].quantile(0.25)
Q3 = df['CO_Concentration'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_count = df[(df['CO_Concentration'] < lower_bound) | (df['CO_Concentration'] > upper_bound)].shape[0]
print(f"Number of outliers detected: {outliers_count}")

median_non_outlier = df[(df['CO_Concentration'] >= lower_bound) & (df['CO_Concentration'] <= upper_bound)]['CO_Concentration'].median()
df.loc[(df['CO_Concentration'] < lower_bound) | (df['CO_Concentration'] > upper_bound), 'CO_Concentration'] = median_non_outlier

print("Outliers have been corrected.")

display(df.head())

Number of outliers detected: 0
Outliers have been corrected.


,Date,CO_Concentration
0,2025-08-24,0.018319
1,2025-08-25,0.020895
2,2025-08-26,0.000000
3,2025-08-27,0.026487
4,2025-08-28,0.010172


## 9. Instalasi TSFEL

In [12]:
!pip install tsfel

## 10. Menyiapkan Data Sinyal

Data CO_Concentration kemudian disiapkan sebagai sinyal.

Program terlebih dahulu memeriksa apakah variabel signal_data sudah tersedia.

Jika belum tersedia, program mengambil data dari kolom:

CO_Concentration

Nilai kosong dihapus menggunakan dropna(), kemudian data diubah menjadi tipe float dan dibuat menjadi list.

Dengan demikian, signal_data menjadi input utama untuk proses ekstraksi fitur.

Notebook menghasilkan Jumlah 365 sampel, Artinya terdapat 365 nilai CO yang digunakan sebagai satu deret waktu.

## 11. Menentukan Parameter Ekstraksi

Program menentukan tiga parameter utama:

- fs = 1
- window_size = len(signal_data)
- overlap = 0
- fs = 1

fs adalah sampling frequency.

Nilai 1 digunakan karena satu sampel dianggap mewakili satu hari.

Ukuran window dibuat sama dengan seluruh panjang data, yaitu 365 sampel. Dengan demikian, seluruh data CO digunakan sebagai satu window.

overlap
overlap = 0

Nilai overlap 0 berarti tidak ada bagian window yang tumpang tindih dengan window lainnya. Parameter yang digunakan pada notebook adalah:

- Jumlah sampel = 365
- Sampling frequency = 1
- Window size = 365
- Overlap = 0

In [13]:
import tsfel

# MENYIAPKAN DATA SINYAL
if "signal_data" not in globals():

    if "df" in globals() and "CO_Concentration" in df.columns:
        signal_data = df["CO_Concentration"].dropna().astype(float).tolist()
    else:
        raise NameError(
            "signal_data belum tersedia. "
            "Jalankan preprocessing CO_lenteng.ipynb terlebih dahulu."
        )

print("Jumlah sampel:", len(signal_data))
print("Contoh data:", signal_data[:5])

# PARAMETER
fs = 1
window_size = len(signal_data)
overlap = 0

print("Sampling frequency :", fs)
print("Window size        :", window_size)
print("Overlap             :", overlap)


Jumlah sampel: 365
Contoh data: [0.018319077789783478, 0.020895369350910187, 0.0, 0.026486512273550034, 0.010171574540436268]
Sampling frequency : 1
Window size        : 365
Overlap             : 0


## 12. Ekstraksi Fitur Statistical

Domain pertama adalah Statistical.

Pada tahap ini TSFEL menganalisis sinyal dan menghitung fitur statistik.

Contoh fitur yang muncul pada hasil antara lain:

* Absolute energy
* Average power
* ECDF
* Mean
* Mean absolute deviation
* Median
* Median absolute deviation
* Minimum
* Peak to peak distance
* Root mean square
* Skewness
* Standard deviation
* Variance

Hasil ekstraksi kemudian disimpan pada:

features_statistical


In [14]:
# STATISTICAL

cfg_statistical = tsfel.get_features_by_domain("statistical")

features_statistical = tsfel.time_series_features_extractor(
    cfg_statistical,
    signal_data,
    fs=fs,
    window_size=window_size,
    overlap=overlap,
    verbose=0
)

print("Jumlah kolom hasil Statistical:", features_statistical.shape[1])
display(features_statistical)


Jumlah kolom hasil Statistical: 31


,0_Absolute energy,0_Average power,0_ECDF Percentile Count_0,0_ECDF Percentile Count_1,0_ECDF Percentile_0,0_ECDF Percentile_1,0_ECDF_0,0_ECDF_1,0_ECDF_2,0_ECDF_3,...,0_Mean,0_Mean absolute deviation,0_Median,0_Median absolute deviation,0_Min,0_Peak to peak distance,0_Root mean square,0_Skewness,0_Standard deviation,0_Variance
0,0.105205,0.000289,73.0,292.0,0.0,0.024243,0.00274,0.005479,0.008219,0.010959,...,0.013046,0.010146,0.015049,0.010463,0.0,0.032519,0.016977,0.009241,0.010865,0.000118


## 13. Ekstraksi Fitur Temporal

Domain kedua adalah Temporal.

Fitur temporal digunakan untuk menggambarkan karakteristik perubahan sinyal terhadap waktu.

Hasil notebook antara lain menampilkan:

* Area under the curve
* Autocorrelation
* Centroid
* Mean absolute diff
* Mean diff
* Median absolute diff
* Median diff
* Negative turning points
* Neighbourhood peaks
* Positive turning points
* Signal distance
* Slope
* Sum absolute diff
* Zero crossing rate

Hasilnya disimpan pada:

features_temporal

In [15]:
# TEMPORAL

cfg_temporal = tsfel.get_features_by_domain("temporal")

features_temporal = tsfel.time_series_features_extractor(
    cfg_temporal,
    signal_data,
    fs=fs,
    window_size=window_size,
    overlap=overlap,
    verbose=0
)

print("Jumlah kolom hasil Temporal:", features_temporal.shape[1])
display(features_temporal)


Jumlah kolom hasil Temporal: 14


,0_Area under the curve,0_Autocorrelation,0_Centroid,0_Mean absolute diff,0_Mean diff,0_Median absolute diff,0_Median diff,0_Negative turning points,0_Neighbourhood peaks,0_Positive turning points,0_Signal distance,0_Slope,0_Sum absolute diff,0_Zero crossing rate
0,4.736215,3.0,215.810494,0.007737,0.000039,0.00425,0.0,91.0,16.0,111.0,364.02222,0.000034,2.816189,94.0


## 14. Ekstraksi Fitur Spectral

Domain ketiga adalah Spectral.

Pada bagian ini terdapat perlakuan khusus. Program membuat daftar kata kunci:

problematic_keywords = [
    'coefficient',
    'level',
    'band',
    'filter',
    'lpcc',
    'mfcc',
    'wavelet'
]

Kemudian fitur tertentu yang memiliki kata kunci tersebut dihapus dari konfigurasi.

Tujuan bagian tersebut adalah menghindari fitur tertentu yang dianggap bermasalah pada proses ekstraksi dalam notebook.

Setelah konfigurasi disesuaikan, proses ekstraksi dilakukan:

Hasil yang muncul antara lain:

* Fundamental frequency
* Human range energy
* Max power spectrum
* Maximum frequency
* Median frequency
* Spectral centroid
* Spectral decrease
* Spectral distance
* Spectral entropy
* Spectral kurtosis
* Spectral positive turning points
* Spectral roll-off
* Spectral roll-on
* Spectral skewness
* Spectral slope
* Spectral spread
* Spectral variation

Hasil disimpan pada:

features_spectral

In [16]:
# SPECTRAL

cfg_spectral = tsfel.get_features_by_domain("spectral")

problematic_keywords = ['coefficient', 'level', 'band', 'filter', 'lpcc', 'mfcc', 'wavelet']

if 'spectral' in cfg_spectral:
    features_to_check = list(cfg_spectral['spectral'].keys())
    for feature_name in features_to_check:
        if any(keyword in feature_name.lower() for keyword in problematic_keywords):
            if feature_name in cfg_spectral['spectral']:
                del cfg_spectral['spectral'][feature_name]

features_spectral = tsfel.time_series_features_extractor(
    cfg_spectral,
    signal_data,
    fs=fs,
    window_size=window_size,
    overlap=overlap,
    verbose=0
)

print("Jumlah kolom hasil Spectral:", features_spectral.shape[1])
display(features_spectral)


Jumlah kolom hasil Spectral: 17


,0_Fundamental frequency,0_Human range energy,0_Max power spectrum,0_Maximum frequency,0_Median frequency,0_Spectral centroid,0_Spectral decrease,0_Spectral distance,0_Spectral entropy,0_Spectral kurtosis,0_Spectral positive turning points,0_Spectral roll-off,0_Spectral roll-on,0_Spectral skewness,0_Spectral slope,0_Spectral spread,0_Spectral variation
0,0.00274,0.0,48.334387,0.457534,0.150685,0.181503,-0.867787,-846.94218,0.753904,1.812777,57.0,0.457534,0.0,0.453955,-0.01769,0.161333,0.212259


## 15. Ekstraksi Fitur Fractal

Domain terakhir adalah Fractal.

Domain Fractal digunakan sebagai bagian terakhir dari ekstraksi fitur TSFEL dalam program.

Notebook mendefinisikan domain ini sebagai domain yang terdiri dari 6 feature function.

In [17]:
# FRACTAL

cfg_fractal = tsfel.get_features_by_domain("fractal")

features_fractal = tsfel.time_series_features_extractor(
    cfg_fractal,
    signal_data,
    fs=fs,
    window_size=window_size,
    overlap=overlap,
    verbose=0
)

print("Jumlah kolom hasil Fractal:", features_fractal.shape[1])
display(features_fractal)


Jumlah kolom hasil Fractal: 6


,0_Detrended fluctuation analysis,0_Higuchi fractal dimension,0_Hurst exponent,0_Maximum fractal length,0_Multiscale entropy,0_Petrosian fractal dimension
0,0.840351,1.928614,0.749593,0.467541,0.978446,1.041527


## 16. Menyimpan Hasil Berdasarkan Domain

Setelah empat proses ekstraksi selesai, hasilnya disimpan dalam sebuah dictionary.
Dictionary tersebut digunakan untuk menyimpan hasil dari setiap domain secara terpisah.

Kemudian program melakukan perulangan:

Tujuannya adalah menampilkan hasil ekstraksi satu per satu berdasarkan domain.

Dengan cara ini, pengguna dapat melihat hasil Statistical, Temporal, Spectral, dan Fractal secara terpisah.

In [18]:
hasil_tsfel = {
    "Statistical": features_statistical,
    "Temporal": features_temporal,
    "Spectral": features_spectral,
    "Fractal": features_fractal
}

for nama_domain, hasil in hasil_tsfel.items():
    print("\n" + "=" * 70)
    print(nama_domain)
    print("Shape:", hasil.shape)
    display(hasil)



Statistical
Shape: (1, 31)


,0_Absolute energy,0_Average power,0_ECDF Percentile Count_0,0_ECDF Percentile Count_1,0_ECDF Percentile_0,0_ECDF Percentile_1,0_ECDF_0,0_ECDF_1,0_ECDF_2,0_ECDF_3,...,0_Mean,0_Mean absolute deviation,0_Median,0_Median absolute deviation,0_Min,0_Peak to peak distance,0_Root mean square,0_Skewness,0_Standard deviation,0_Variance
0,0.105205,0.000289,73.0,292.0,0.0,0.024243,0.00274,0.005479,0.008219,0.010959,...,0.013046,0.010146,0.015049,0.010463,0.0,0.032519,0.016977,0.009241,0.010865,0.000118



Temporal
Shape: (1, 14)


,0_Area under the curve,0_Autocorrelation,0_Centroid,0_Mean absolute diff,0_Mean diff,0_Median absolute diff,0_Median diff,0_Negative turning points,0_Neighbourhood peaks,0_Positive turning points,0_Signal distance,0_Slope,0_Sum absolute diff,0_Zero crossing rate
0,4.736215,3.0,215.810494,0.007737,0.000039,0.00425,0.0,91.0,16.0,111.0,364.02222,0.000034,2.816189,94.0



Spectral
Shape: (1, 17)


,0_Fundamental frequency,0_Human range energy,0_Max power spectrum,0_Maximum frequency,0_Median frequency,0_Spectral centroid,0_Spectral decrease,0_Spectral distance,0_Spectral entropy,0_Spectral kurtosis,0_Spectral positive turning points,0_Spectral roll-off,0_Spectral roll-on,0_Spectral skewness,0_Spectral slope,0_Spectral spread,0_Spectral variation
0,0.00274,0.0,48.334387,0.457534,0.150685,0.181503,-0.867787,-846.94218,0.753904,1.812777,57.0,0.457534,0.0,0.453955,-0.01769,0.161333,0.212259



Fractal
Shape: (1, 6)


,0_Detrended fluctuation analysis,0_Higuchi fractal dimension,0_Hurst exponent,0_Maximum fractal length,0_Multiscale entropy,0_Petrosian fractal dimension
0,0.840351,1.928614,0.749593,0.467541,0.978446,1.041527


## 17. Menggabungkan Seluruh Fitur

Setelah setiap domain diperoleh, seluruh hasil digabungkan, Sebelum digabungkan, index masing-masing DataFrame diatur ulang menggunakan:

reset_index(drop=True)

Hal tersebut dilakukan agar index dari masing-masing hasil ekstraksi konsisten.


program kemudian menghitung jumlah fitur,
Jika jumlahnya bukan 68, program akan menghasilkan ValueError sehingga pengguna mengetahui bahwa terdapat masalah pada proses ekstraksi.

Setelah jumlah fitur dipastikan, nama kolom diubah menjadi:

P1, P2, P3, ..., P68

Tahap terakhir adalah menyimpan hasil ekstraksi ke dalam file CSV.

In [27]:
# gabung
import pandas as pd

hasil_68 = pd.concat(
    [
        features_statistical.reset_index(drop=True),
        features_temporal.reset_index(drop=True),
        features_spectral.reset_index(drop=True),
        features_fractal.reset_index(drop=True)
    ],
    axis=1
)

jumlah_fitur = hasil_68.shape[1]
if jumlah_fitur != 68:
    raise ValueError(
        f"Jumlah fitur bukan 68, tetapi {jumlah_fitur}. "
        "Periksa kembali hasil ekstraksi TSFEL."
    )

hasil_68.columns = [f"P{i}" for i in range(1, 69)]


hasil_68.to_csv("CO_lenteng_TSFEL_P1_P68.csv", index=False)

print("CSV berhasil disimpan!")
print("Jumlah fitur:", jumlah_fitur)
print("Jumlah baris:", hasil_68.shape[0])
print("Shape keseluruhan:", hasil_68.shape)

display(hasil_68)

CSV berhasil disimpan!
Jumlah fitur: 68
Jumlah baris: 1
Shape keseluruhan: (1, 68)


,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,...,P59,P60,P61,P62,P63,P64,P65,P66,P67,P68
0,0.105205,0.000289,73.0,292.0,0.0,0.024243,0.00274,0.005479,0.008219,0.010959,...,0.453955,-0.01769,0.161333,0.212259,0.840351,1.928614,0.749593,0.467541,0.978446,1.041527
